# 01 - Data Loading and Merging

**Author:** Günther (with Claude's assistance)  
**Purpose:** Load crash data from NHTSA and Waymo, merge them together, and prepare for analysis

---

## What This Notebook Does

We're working with crash data from two sources:

1. **NHTSA (National Highway Traffic Safety Administration):** The government agency that collects crash reports from all autonomous vehicle companies. This is our primary data source.

2. **Waymo Safety Impact Data Hub:** Waymo's own published data about their crashes. This includes some extra information not in the NHTSA data, like delta-V (impact force) and zip codes.

This notebook:
1. Loads the raw data files
2. Filters for Waymo-only crashes
3. Removes duplicate report versions (keeps only the latest)
4. Merges in useful columns from Waymo's data
5. Saves a clean, merged dataset for analysis

---

## Important Background: The June 16, 2025 Reporting Change

On June 16, 2025, NHTSA changed the rules for which crashes autonomous vehicles must report:

- **Before June 16:** Companies reported ALL crashes, even minor fender-benders
- **After June 16:** Companies only report more serious crashes (injuries, tow-aways, or significant damage)

This means we have two different datasets with different reporting standards:
- `nhtsa_ads_prior_june16` = All crashes (comprehensive)
- `nhtsa_ads_post_june16` = Only serious crashes (filtered by NHTSA rules)

We'll handle this difference in the next notebook by creating a filter.

---

## Step 0: Import Libraries

In [ ]:
import pandas as pd
import os

## Step 1: Define File Paths

Update these paths when new data arrives:
- **NHTSA data:** Released on the 15th of each month
- **Waymo Safety Hub data:** Updated quarterly (every 3 months)

The filenames include the download date (YYYYMMDD) so we can track which version we're using.

In [ ]:
# =============================================================================
# INPUT FILE PATHS - Update these when new data arrives
# =============================================================================

# NHTSA Standing General Order data
# Downloaded from: https://www.nhtsa.gov/laws-regulations/standing-general-order-crash-reporting
# - Prior: Crashes BEFORE June 16, 2025 (old reporting rules - all crashes)
# - Post: Crashes AFTER June 16, 2025 (new reporting rules - serious crashes only)
NHTSA_PRIOR_PATH = "../data/nhtsa_ads_prior_june16_20250129.csv"
NHTSA_POST_PATH = "../data/nhtsa_ads_post_june16_20250129.csv"

# Waymo Safety Impact Data Hub - CSV2 file
# Downloaded from: https://waymo.com/safety/data/
# This file contains delta-V (impact force) data and zip codes
WAYMO_HUB_PATH = "../data/waymo_safety_hub_csv2_20250129.csv"

# =============================================================================
# OUTPUT FILE PATH
# =============================================================================

# This is where we'll save our merged, cleaned dataset
OUTPUT_PATH = "../data/waymo_crashes_merged_20250129.csv"

In [ ]:
# =============================================================================
# Verify that all input files exist before we try to load them
# =============================================================================

print("Checking if input files exist...")
print()

files_to_check = [
    ("NHTSA Prior (before June 16)", NHTSA_PRIOR_PATH),
    ("NHTSA Post (after June 16)", NHTSA_POST_PATH),
    ("Waymo Safety Hub CSV2", WAYMO_HUB_PATH),
]

all_found = True
for name, path in files_to_check:
    if os.path.exists(path):
        print(f"[OK] {name}")
    else:
        print(f"[MISSING] {name}")
        print(f"          Expected at: {path}")
        all_found = False

if not all_found:
    print()
    print("WARNING: Some files are missing!")
    print("Please check the file paths in Step 1 above.")
else:
    print()
    print("All files found. Ready to proceed.")

## Step 2: Load NHTSA Data

The NHTSA data contains crash reports from ALL autonomous vehicle companies (Waymo, Cruise, Tesla, etc.). We'll load it and then filter for Waymo only.

**Technical note:** We read Zip Code as a string (text) instead of a number. This prevents Python from dropping leading zeros (e.g., keeping "02134" instead of converting it to 2134).

In [ ]:
# =============================================================================
# Load NHTSA data from BEFORE June 16, 2025
# This dataset uses the OLD reporting rules (all crashes reported)
# =============================================================================

# Read the CSV file
# dtype={'Zip Code': str} tells pandas to treat Zip Code as text, not numbers
nhtsa_prior_all = pd.read_csv(NHTSA_PRIOR_PATH, dtype={'Zip Code': str})

print(f"Loaded NHTSA Prior data")
print(f"  Total rows (all companies): {len(nhtsa_prior_all):,}")
print(f"  Columns: {len(nhtsa_prior_all.columns)}")

# See which companies are in the data
print()
print("Companies in the data:")
print(nhtsa_prior_all['Reporting Entity'].value_counts())

In [ ]:
# =============================================================================
# Load NHTSA data from AFTER June 16, 2025
# This dataset uses the NEW reporting rules (only serious crashes reported)
# =============================================================================

nhtsa_post_all = pd.read_csv(NHTSA_POST_PATH, dtype={'Zip Code': str})

print(f"Loaded NHTSA Post data")
print(f"  Total rows (all companies): {len(nhtsa_post_all):,}")
print(f"  Columns: {len(nhtsa_post_all.columns)}")

# See which companies are in the data
print()
print("Companies in the data:")
print(nhtsa_post_all['Reporting Entity'].value_counts())

## Step 3: Filter for Waymo Only

Since we're analyzing Waymo specifically, we filter out all other companies.

In [ ]:
# =============================================================================
# Keep only Waymo crashes
# =============================================================================

# Filter: keep rows where 'Reporting Entity' equals 'Waymo LLC'
# .copy() creates a fresh copy so we don't accidentally modify the original
waymo_prior = nhtsa_prior_all[nhtsa_prior_all['Reporting Entity'] == 'Waymo LLC'].copy()
waymo_post = nhtsa_post_all[nhtsa_post_all['Reporting Entity'] == 'Waymo LLC'].copy()

print(f"Waymo crashes before June 16: {len(waymo_prior):,}")
print(f"Waymo crashes after June 16:  {len(waymo_post):,}")
print(f"Total Waymo crashes:          {len(waymo_prior) + len(waymo_post):,}")

## Step 4: Remove Duplicate Report Versions

Sometimes Waymo submits updated versions of a crash report (corrections, additional information, etc.). Each version has a different `Report Version` number. We only want the most recent version of each crash.

**Example:**
- Report ID `30270-11087`, Version 1 (original)
- Report ID `30270-11087`, Version 2 (updated) ← Keep this one

We sort by version number and keep only the last (highest) version for each Report ID.

In [ ]:
# =============================================================================
# Remove duplicate versions from PRIOR data
# =============================================================================

print("Removing duplicate report versions from Prior data...")
print(f"  Before: {len(waymo_prior):,} rows")

# Step 1: Sort by Report ID and Report Version
# This puts version 1, 2, 3... in order for each crash
waymo_prior = waymo_prior.sort_values(['Report ID', 'Report Version'])

# Step 2: Keep only the last row for each Report ID (highest version)
# drop_duplicates removes duplicate Report IDs, keeping the last one
waymo_prior = waymo_prior.drop_duplicates(subset='Report ID', keep='last')

print(f"  After:  {len(waymo_prior):,} rows")
print(f"  Removed {len(nhtsa_prior_all[nhtsa_prior_all['Reporting Entity'] == 'Waymo LLC']) - len(waymo_prior)} duplicate versions")

In [ ]:
# =============================================================================
# Remove duplicate versions from POST data
# =============================================================================

print("Removing duplicate report versions from Post data...")
print(f"  Before: {len(waymo_post):,} rows")

waymo_post = waymo_post.sort_values(['Report ID', 'Report Version'])
waymo_post = waymo_post.drop_duplicates(subset='Report ID', keep='last')

print(f"  After:  {len(waymo_post):,} rows")
print(f"  Removed {len(nhtsa_post_all[nhtsa_post_all['Reporting Entity'] == 'Waymo LLC']) - len(waymo_post)} duplicate versions")

In [ ]:
# =============================================================================
# Verify: Each Report ID should now appear exactly once
# =============================================================================

prior_unique = waymo_prior['Report ID'].nunique()
post_unique = waymo_post['Report ID'].nunique()

print("Verification:")
print(f"  Prior data: {len(waymo_prior)} rows, {prior_unique} unique Report IDs")
print(f"  Post data:  {len(waymo_post)} rows, {post_unique} unique Report IDs")

if len(waymo_prior) == prior_unique and len(waymo_post) == post_unique:
    print()
    print("  [OK] Each crash appears exactly once. No duplicates remain.")
else:
    print()
    print("  [WARNING] Something went wrong - duplicates may still exist.")

## Step 5: Load Waymo Safety Hub Data

Waymo publishes their own crash data on their Safety Impact Data Hub. This includes some information not available in the raw NHTSA data:

- **Delta-V less than 1 mph:** Whether the impact force was very minor (like bumping a shopping cart)
- **Zip Code:** NHTSA redacts this, but Waymo provides it
- **Location:** City name (San Francisco, Phoenix, Los Angeles, etc.)

The delta-V information is particularly important for our filtering in the next notebook.

In [ ]:
# =============================================================================
# Load Waymo Safety Hub data
# =============================================================================

# Read with Zip Code as string to preserve leading zeros
waymo_hub = pd.read_csv(WAYMO_HUB_PATH, dtype={'Zip Code': str})

print(f"Loaded Waymo Safety Hub data")
print(f"  Total rows: {len(waymo_hub):,}")
print(f"  Columns: {len(waymo_hub.columns)}")

# Show all column names
print()
print("Available columns:")
for col in waymo_hub.columns:
    print(f"  - {col}")

In [ ]:
# =============================================================================
# Check the delta-V field (this is key for our later filtering)
# =============================================================================

# Delta-V = change in velocity during a crash
# Lower delta-V = less severe impact
# < 1 mph is a VERY minor impact (like barely tapping something)

deltav_col = 'Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH'

print("Delta-V distribution in Waymo Hub data:")
print(waymo_hub[deltav_col].value_counts(dropna=False))
print()
print("Interpretation:")
print("  True = Very minor impact (delta-V < 1 mph)")
print("  False = More significant impact (delta-V >= 1 mph)")

## Step 6: Merge Waymo Hub Data into NHTSA Data

Now we'll add the useful columns from the Waymo Hub to our NHTSA data. We match records using the Report ID (called `SGO Report ID` in the Hub data).

Not all NHTSA records will have a match in the Hub (different time periods, different data releases). Those will have blank values in the Hub columns, which is fine.

In [ ]:
# =============================================================================
# Prepare Waymo Hub data for merging
# =============================================================================

# Select which columns we want to bring in from the Hub
# We'll rename them with 'hub_' prefix to show they came from Waymo Hub

hub_columns = {
    'SGO Report ID': 'Report ID',  # This is the matching key (same as NHTSA Report ID)
    'Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH': 'hub_delta_v_less_than_1mph',
    'Zip Code': 'hub_zip_code',
    'Location': 'hub_location',
    'Crash Type': 'hub_crash_type',
    'Is Police-Reported': 'hub_is_police_reported',
    'Is Any-Injury-Reported': 'hub_is_any_injury_reported',
}

# Keep only columns that exist in this version of the Hub data
available_columns = {k: v for k, v in hub_columns.items() if k in waymo_hub.columns}

# Create a subset with just the columns we need, renamed
hub_for_merge = waymo_hub[list(available_columns.keys())].copy()
hub_for_merge = hub_for_merge.rename(columns=available_columns)

print("Columns prepared for merge:")
for col in hub_for_merge.columns:
    print(f"  - {col}")

In [ ]:
# =============================================================================
# Merge Hub data into the PRIOR (before June 16) dataset
# =============================================================================

print("Merging Hub data into Prior dataset...")
print(f"  Prior crashes: {len(waymo_prior):,}")
print(f"  Hub records:   {len(hub_for_merge):,}")

# Left merge: keep ALL Prior records, add Hub data where available
# Records without a Hub match will have NaN (blank) in the hub_ columns
waymo_prior = waymo_prior.merge(hub_for_merge, on='Report ID', how='left')

# Count how many matched
matched = waymo_prior['hub_delta_v_less_than_1mph'].notna().sum()
unmatched = waymo_prior['hub_delta_v_less_than_1mph'].isna().sum()

print()
print(f"Results:")
print(f"  Matched with Hub data:     {matched:,} ({100*matched/len(waymo_prior):.1f}%)")
print(f"  No Hub match (blank):      {unmatched:,} ({100*unmatched/len(waymo_prior):.1f}%)")

In [ ]:
# =============================================================================
# Merge Hub data into the POST (after June 16) dataset
# =============================================================================

print("Merging Hub data into Post dataset...")
print(f"  Post crashes:  {len(waymo_post):,}")
print(f"  Hub records:   {len(hub_for_merge):,}")

waymo_post = waymo_post.merge(hub_for_merge, on='Report ID', how='left')

matched = waymo_post['hub_delta_v_less_than_1mph'].notna().sum()
unmatched = waymo_post['hub_delta_v_less_than_1mph'].isna().sum()

print()
print(f"Results:")
print(f"  Matched with Hub data:     {matched:,} ({100*matched/len(waymo_post):.1f}%)")
print(f"  No Hub match (blank):      {unmatched:,} ({100*unmatched/len(waymo_post):.1f}%)")

## Step 7: Add Data Period Labels

We'll add a column to indicate which reporting period each crash belongs to. This makes it easy to filter or compare later.

In [ ]:
# =============================================================================
# Add a column indicating the data period
# =============================================================================

# Add labels
waymo_prior['data_period'] = 'prior_june16'  # Old rules: all crashes reported
waymo_post['data_period'] = 'post_june16'    # New rules: only serious crashes

print("Added 'data_period' column:")
print(f"  Prior data labeled as: 'prior_june16'")
print(f"  Post data labeled as:  'post_june16'")

## Step 8: Combine Both Datasets

Now we combine the prior and post datasets into one complete dataset. This gives us all Waymo crashes in one place.

**Note:** The prior and post NHTSA files have slightly different columns (the format changed with the new rules). We'll keep all columns from both.

In [ ]:
# =============================================================================
# Combine prior and post datasets
# =============================================================================

# pd.concat stacks the dataframes on top of each other
# ignore_index=True gives us fresh row numbers (0, 1, 2, ...)
waymo_combined = pd.concat([waymo_prior, waymo_post], ignore_index=True)

print("Combined dataset:")
print(f"  Prior crashes: {len(waymo_prior):,}")
print(f"  Post crashes:  {len(waymo_post):,}")
print(f"  Total:         {len(waymo_combined):,}")
print()
print(f"  Columns: {len(waymo_combined.columns)}")

In [ ]:
# =============================================================================
# Verify the combined data
# =============================================================================

print("Data period breakdown:")
print(waymo_combined['data_period'].value_counts())
print()

# Make sure no Report IDs appear in both prior and post
# (They shouldn't - each crash happened on one specific date)
prior_ids = set(waymo_prior['Report ID'])
post_ids = set(waymo_post['Report ID'])
overlap = prior_ids.intersection(post_ids)

if len(overlap) == 0:
    print("[OK] No Report IDs appear in both prior and post data.")
else:
    print(f"[WARNING] {len(overlap)} Report IDs appear in both datasets!")
    print(f"          This shouldn't happen. Check the data.")

## Step 9: Save the Merged Dataset

We'll save the combined dataset as a CSV file for use in the next notebooks.

In [ ]:
# =============================================================================
# Save the merged dataset
# =============================================================================

# Sort by incident date for easier browsing
if 'Incident Date' in waymo_combined.columns:
    waymo_combined = waymo_combined.sort_values('Incident Date')

# Save to CSV
# index=False prevents pandas from adding an extra index column
waymo_combined.to_csv(OUTPUT_PATH, index=False)

print(f"[OK] Saved merged dataset to: {OUTPUT_PATH}")
print(f"     {len(waymo_combined):,} crashes")
print(f"     {len(waymo_combined.columns)} columns")

In [ ]:
# =============================================================================
# Also save prior and post separately (useful for the filtering notebook)
# =============================================================================

# Generate output paths based on the main output path
prior_output = OUTPUT_PATH.replace('_merged_', '_prior_merged_')
post_output = OUTPUT_PATH.replace('_merged_', '_post_merged_')

waymo_prior.to_csv(prior_output, index=False)
print(f"[OK] Saved prior data to: {prior_output}")
print(f"     {len(waymo_prior):,} crashes")

waymo_post.to_csv(post_output, index=False)
print(f"[OK] Saved post data to: {post_output}")
print(f"     {len(waymo_post):,} crashes")

## Step 10: Summary

Let's review what we created.

In [ ]:
# =============================================================================
# Final Summary
# =============================================================================

print("="*70)
print("DATA LOADING AND MERGING COMPLETE")
print("="*70)
print()
print("DATASETS CREATED:")
print()
print(f"1. Combined dataset: {OUTPUT_PATH}")
print(f"   - {len(waymo_combined):,} total Waymo crashes")
print(f"   - {len(waymo_combined.columns)} columns")
print()
print(f"2. Prior only (before June 16): {prior_output}")
print(f"   - {len(waymo_prior):,} crashes")
print(f"   - Uses OLD reporting rules (all crashes)")
print()
print(f"3. Post only (after June 16): {post_output}")
print(f"   - {len(waymo_post):,} crashes")
print(f"   - Uses NEW reporting rules (serious crashes only)")
print()
print("NEW COLUMNS ADDED FROM WAYMO HUB:")
hub_cols = [col for col in waymo_combined.columns if col.startswith('hub_')]
for col in hub_cols:
    print(f"   - {col}")
print()
print("NEXT STEP:")
print("   Run 02_amendment3_filter_creation.ipynb to create filters for")
print("   apples-to-apples comparison across the June 16 reporting change.")

In [ ]:
# =============================================================================
# Show all columns in the final dataset
# =============================================================================

print("ALL COLUMNS IN THE MERGED DATASET:")
print()
for i, col in enumerate(waymo_combined.columns, 1):
    # Mark new columns we added
    if col.startswith('hub_'):
        print(f"  {i:3}. {col}  [NEW - from Waymo Hub]")
    elif col == 'data_period':
        print(f"  {i:3}. {col}  [NEW - added by this notebook]")
    else:
        print(f"  {i:3}. {col}")